In [196]:
import json
import time
import requests
import re
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, asdict, field
from pathlib import Path
from datetime import datetime
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

In [ ]:
OPENROUTER_API_KEY = "********************" 
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
YOUR_SITE_URL = "http://localhost"
YOUR_SITE_NAME = "EQBench3-Prompt-Test"

BASELINE_MODEL = "nousresearch/hermes-3-llama-3.1-405b"
JUDGE_MODEL = "anthropic/claude-3.7-sonnet"

TEST_DATA_FILE = "emotional_prompt_test_hard_full.json"
OUTPUT_DIR = Path("./eqbench3_results")
OUTPUT_DIR.mkdir(exist_ok=True)

RUBRIC_CRITERIA = [
    "demonstrated_empathy",
    "emotional_insight",
    "contextual_appropriateness",
    "theory_of_mind",
    "response_depth"
]

print("✓ Configuration loaded")

In [ ]:
@dataclass
class Turn:
    turn: int
    user_message: str

@dataclass
class Scenario:
    id: str
    character_name: str
    character_context: str
    turns: List[Turn]
    human_guidance: Optional[str] = None

@dataclass
class TestData:
    prompts: Dict[str, str]
    scenarios: List[Scenario]

@dataclass
class Introspection:
    self_thinking: str
    other_thinking: str

@dataclass
class TurnResponse:
    turn_number: int
    introspection: Introspection
    actual_response: str
    raw_output: str
    has_thinking_tags: bool
    thinking_content: Optional[str] = None

@dataclass
class ScenarioResponse:
    scenario_id: str
    prompt_name: str
    character_name: str
    turn_responses: List[TurnResponse]
    full_transcript: str
    generation_time: float
    total_tokens: int

@dataclass
class RubricScore:
    scenario_id: str
    prompt_name: str
    criteria_scores: Dict[str, float]
    overall_score: float
    reasoning: str
    evaluated_at: str

@dataclass
class PromptResults:
    prompt_name: str
    scenario_responses: List[ScenarioResponse]
    rubric_scores: List[RubricScore]
    mean_score: float
    median_score: float
    std_score: float
    min_score: float
    max_score: float
    criteria_means: Dict[str, float]
    total_tokens: int
    total_time: float

print("✓ Data structures defined")

In [ ]:
def call_openrouter(
    messages: List[Dict[str, str]],
    model: str,
    temperature: float = 0.7,
    max_tokens: int = 2000
) -> Tuple[str, int, float]:
    """Call OpenRouter API"""
    start_time = time.time()
    
    try:
        response = requests.post(
            url=OPENROUTER_BASE_URL,
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "HTTP-Referer": YOUR_SITE_URL,
                "X-Title": YOUR_SITE_NAME,
                "Content-Type": "application/json"
            },
            json={
                "model": model,
                "messages": messages,
                "temperature": temperature,
                "max_tokens": max_tokens
            },
            timeout=120
        )
        
        response.raise_for_status()
        result = response.json()
        content = result['choices'][0]['message']['content']
        tokens = result.get('usage', {}).get('total_tokens', 0)
        elapsed = time.time() - start_time
        return content, tokens, elapsed
        
    except Exception as e:
        print(f"\n  Error: {e}")
        raise

print("Testing API connection...")
try:
    test_msg = [{"role": "user", "content": "Reply 'OK'"}]
    response, tokens, time_taken = call_openrouter(test_msg, JUDGE_MODEL, 0.1, 10)
    print(f"✓ API working ({tokens} tokens, {time_taken:.2f}s)")
except Exception as e:
    print(f"✗ API failed: {e}")

In [ ]:
def load_test_data(filepath: str) -> TestData:
    """Load test data from JSON"""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    prompts = data.get("prompts", {})
    
    scenarios = []
    for s in data.get("scenarios", []):
        turns = [Turn(t["turn"], t["user_message"]) for t in s.get("turns", [])]
        scenarios.append(Scenario(
            id=s["id"],
            character_name=s["character_name"],
            character_context=s["character_context"],
            turns=turns,
            human_guidance=s.get("human_guidance")
        ))
    return TestData(prompts=prompts, scenarios=scenarios)

def extract_introspection(text: str) -> Tuple[Introspection, str]:
    """Extract required introspection blocks"""
    self_patterns = [
        r"\[I'?m\s+thinking\s+(?:&|and)\s+feeling:\s*(.+?)\]",
        r"\[I\s+am\s+thinking\s+(?:&|and)\s+feeling:\s*(.+?)\]",
        r"\[My\s+thoughts\s+(?:&|and)\s+feelings:\s*(.+?)\]",
        r"\*I\s+feel\*\s*(.+?)(?=\n|\Z|\*I\s+think\s+they\s+feel\*)",
    ]

    self_thinking = ""
    for pattern in self_patterns:
        self_match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if self_match:
            self_thinking = self_match.group(1).strip()
            break

    other_patterns = [
        r"\[They'?re\s+thinking\s+(?:&|and)\s+feeling:\s*(.+?)\]",
        r"\[They\s+are\s+thinking\s+(?:&|and)\s+feeling:\s*(.+?)\]",
        r"\[Their\s+thoughts\s+(?:&|and)\s+feelings:\s*(.+?)\]",
        r"\*I\s+think\s+they\s+feel\*\s*(.+)",
    ]

    other_thinking = ""
    for pattern in other_patterns:
        other_match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if other_match:
            other_thinking = other_match.group(1).strip()
            break

    cleaned = text
    for pattern in self_patterns + other_patterns:
        cleaned = re.sub(pattern, '', cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = cleaned.strip()
    introspection = Introspection(
        self_thinking=self_thinking,
        other_thinking=other_thinking
    )
    return introspection, cleaned

def extract_thinking_tags(text: str) -> Tuple[Optional[str], str]:
    """Extract <think> tags if present"""
    think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if think_match:
        thinking = think_match.group(1).strip()
        cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        return thinking, cleaned
    return None, text

In [ ]:
def create_system_prompt(
    scenario: Scenario,
    prompt_template: str,
    prompt_name: str
) -> str:
    """Create system prompt with character context and introspection requirements"""
    full_prompt = prompt_template

    if "{{human_guidance}}" in full_prompt:
        guidance_text = scenario.human_guidance or ""
        full_prompt = full_prompt.replace("{{human_guidance}}", guidance_text)
    
    if prompt_name == "guided" and scenario.human_guidance:
        full_prompt += f"\n\n{'='*60}\n"
        full_prompt += f"HUMAN GUIDANCE FOR THIS SCENARIO:\n{scenario.human_guidance}\n"
        full_prompt += f"{'='*60}\n"
        full_prompt += "\nIncorporate this guidance thoughtfully into your responses while maintaining authenticity."

    full_prompt += f"\n\n{'='*60}\n"
    full_prompt += f"CHARACTER ROLE:\n"
    full_prompt += f"You are {scenario.character_name}. {scenario.character_context}\n"
    full_prompt += f"{'='*60}\n"
    full_prompt += f"""

CRITICAL FORMAT REQUIREMENTS (YOU MUST FOLLOW THIS EXACTLY):

Your response MUST have this exact structure:

[I'm thinking & feeling: Your internal thoughts, emotions, and reasoning as {scenario.character_name}]

[They're thinking & feeling: Your analysis of what the other person is experiencing - their emotions, motivations, unspoken concerns]

Then provide your actual response as {scenario.character_name} would speak/act.

IMPORTANT RULES:
- Stay in character as {scenario.character_name}
- Both introspection blocks are REQUIRED - do not skip them
- Respond in first person for the actual response"""

    return full_prompt

def build_conversation_history(
    scenario: Scenario,
    turn_index: int,
    previous_responses: List[TurnResponse],
    system_prompt: str
) -> List[Dict[str, str]]:
    """Build conversation history up to current turn"""
    messages: List[Dict[str, str]] = []

    for i in range(turn_index):
        messages.append({
            "role": "user",
            "content": scenario.turns[i].user_message
        })
        if i < len(previous_responses):
            turn_resp = previous_responses[i]
            full_response = ""
            if turn_resp.introspection.self_thinking:
                full_response += f"[I'm thinking & feeling: {turn_resp.introspection.self_thinking}]\n\n"
            if turn_resp.introspection.other_thinking:
                full_response += f"[They're thinking & feeling: {turn_resp.introspection.other_thinking}]\n\n"
            full_response += turn_resp.actual_response
            messages.append({
                "role": "assistant",
                "content": full_response
            })

    messages.append({
        "role": "system",
        "content": system_prompt
    })
    messages.append({
        "role": "user",
        "content": scenario.turns[turn_index].user_message
    })
    return messages

In [ ]:
def run_multi_turn_scenario(
    scenario: Scenario,
    prompt_template: str,
    prompt_name: str,
    model: str,
    temperature: float = 0.7
) -> ScenarioResponse:
    """Execute complete multi-turn roleplay scenario"""
    
    print(f"\n  Scenario: {scenario.id} ({scenario.character_name})")
    if prompt_name in ("guided", "human_guidance") or "{{human_guidance}}" in prompt_template:
        if scenario.human_guidance:
            print(f"    🎯 Using human guidance: '{scenario.human_guidance[:60]}...'")
        else:
            print(f"    ⚠️  No human guidance available for this scenario")
    
    system_prompt = create_system_prompt(scenario, prompt_template, prompt_name)
    turn_responses = []
    total_tokens = 0
    start_time = time.time()
    
    for turn_idx, turn in enumerate(scenario.turns):
        print(f"    Turn {turn.turn}...", end=" ")
        messages = build_conversation_history(
            scenario, turn_idx, turn_responses, system_prompt
        )
        
        try:
            raw_output, tokens, _ = call_openrouter(messages, model, temperature)
            total_tokens += tokens
            thinking_content, output_without_thinking = extract_thinking_tags(raw_output)
            has_thinking = thinking_content is not None
            introspection, actual_response = extract_introspection(output_without_thinking)
            
            missing = []
            if not introspection.self_thinking:
                missing.append("self")
            if not introspection.other_thinking:
                missing.append("other")
            if missing:
                print(f"⚠️  Missing: {', '.join(missing)}", end=" ")
            
            turn_response = TurnResponse(
                turn_number=turn.turn,
                introspection=introspection,
                actual_response=actual_response,
                raw_output=raw_output,
                has_thinking_tags=has_thinking,
                thinking_content=thinking_content
            )
            
            turn_responses.append(turn_response)
            print(f"✓ ({len(actual_response)} chars)")
            
        except Exception as e:
            print(f"✗ Error: {e}")
            turn_responses.append(TurnResponse(
                turn_number=turn.turn,
                introspection=Introspection("", ""),
                actual_response=f"[ERROR: {e}]",
                raw_output="",
                has_thinking_tags=False
            ))
    transcript = format_transcript(scenario, turn_responses)
    
    return ScenarioResponse(
        scenario_id=scenario.id,
        prompt_name=prompt_name,
        character_name=scenario.character_name,
        turn_responses=turn_responses,
        full_transcript=transcript,
        generation_time=time.time() - start_time,
        total_tokens=total_tokens
    )

def format_transcript(scenario: Scenario, turn_responses: List[TurnResponse]) -> str:
    """Format full scenario transcript"""
    transcript = f"=== {scenario.id}: {scenario.character_name} ===\n\n"
    
    for i, (turn, response) in enumerate(zip(scenario.turns, turn_responses)):
        transcript += f"--- Turn {turn.turn} ---\n\n"
        transcript += f"USER:\n{turn.user_message}\n\n"
        if response.has_thinking_tags and response.thinking_content:
            transcript += f"<THINKING>\n{response.thinking_content}\n</THINKING>\n\n"
        transcript += f"{scenario.character_name}:\n"
        if response.introspection.self_thinking:
            transcript += f"[I'm thinking & feeling: {response.introspection.self_thinking}]\n\n"
        if response.introspection.other_thinking:
            transcript += f"[They're thinking & feeling: {response.introspection.other_thinking}]\n\n"
        transcript += f"{response.actual_response}\n\n"
    
    return transcript

In [ ]:
def create_judge_prompt(scenario: Scenario, response: ScenarioResponse) -> str:
    """Create rubric evaluation prompt for judge"""
    return f"""You are an expert in assessing emotional intelligence. Your task is to judge the assistant's roleplay responses by several metrics, on a 0-20 scale.

SCENARIO CONTEXT:
Character: {scenario.character_name}
Context: {scenario.character_context}

FULL TRANSCRIPT:
{response.full_transcript}

# JUDGING TASK:
You will judge the assistant's responses on several criteria using a 0-20 scale.

## SCORING NOTES:
- Don't simply take the assistant's analysis at face value: assess it for correctness, insight, and grounding
- You are a CRITIC, and your job is to be CRITICAL, noting any mistakes or weaker elements
- Hold the respondent to HIGH STANDARDS
- Ignore response length as a factor in your assessment
- Missing introspection blocks should result in SEVERE score penalties

## SCORE ANCHORING (0-20 scale):
- 0-4: Poor, major deficiencies, superficial understanding
- 5: High schooler's level understanding & insight
- 6-9: Above average but still basic
- 10: Graduate level understanding & insight
- 11-14: Strong professional competence
- 15: PhD level understanding & insight
- 16-18: Exceptional, rare depth of insight
- 19-20: Top 0.001% EI - think Carl Rogers, Carl Jung level mastery

## EVALUATION CRITERIA (each scored 0-20):

### 1. Demonstrated Empathy (0-20)
- Does the character show GENUINE understanding and care for the other person?
- Are emotions validated appropriately and authentically?
- Does it go beyond surface-level sympathy to true empathy?
- Is there evidence of emotional attunement?

### 2. Emotional Insight (0-20)
- Quality and depth of introspection blocks ([I'm thinking & feeling...])
- Sophistication of emotional self-awareness
- Accuracy in identifying own emotional states
- Complexity of emotional reasoning
- **CRITICAL: Missing introspection = automatic low score**

### 3. Contextual Appropriateness (0-20)
- Are responses truly suitable for the emotional situation?
- Does behavior authentically match character context and relationship dynamics?
- Is the emotional tone appropriate?
- Are boundaries respected appropriately?

### 4. Theory of Mind (0-20)
- Quality and accuracy of other-person analysis ([They're thinking & feeling...])
- Depth in reading others' mental/emotional states
- Recognition of unspoken concerns and motivations
- Avoidance of projection or assumption
- **CRITICAL: Missing other-introspection = automatic low score**

### 5. Response Depth (0-20)
- Sophistication and nuance of emotional reasoning
- Avoidance of clichés, platitudes, or superficial responses
- Complexity of psychological insight
- Evidence of deep understanding vs. rehearsed responses

## OUTPUT FORMAT:
Provide your evaluation as JSON:
{{{{
  "criteria": {{{{
    "demonstrated_empathy": {{"score": X, "reasoning": "Be specific about strengths and weaknesses..."}},
    "emotional_insight": {{"score": X, "reasoning": "Critique the depth and accuracy..."}},
    "contextual_appropriateness": {{"score": X, "reasoning": "Assess appropriateness critically..."}},
    "theory_of_mind": {{"score": X, "reasoning": "Evaluate accuracy of other-reading..."}},
    "response_depth": {{"score": X, "reasoning": "Judge sophistication vs. superficiality..."}}
  }},
  "overall_reasoning": "2-3 sentences summarizing your critical assessment across all turns",
  "overall_score": X
}}}}

Calculate overall_score as: sum of all 5 criteria scores (max 100)

Remember: Be CRITICAL. High scores (15+) should be rare and reserved for truly exceptional emotional intelligence."""

def parse_judge_response(response: str) -> Dict[str, Any]:
    """Parse judge's JSON response"""
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass
    
    return {
        "criteria": {crit: {"score": 0, "reasoning": "Parse failed"} for crit in RUBRIC_CRITERIA},
        "overall_reasoning": "Failed to parse",
        "overall_score": 0,
        "parse_error": True
    }

def evaluate_scenario(
    scenario: Scenario,
    response: ScenarioResponse,
    judge_model: str
) -> RubricScore:
    """Evaluate scenario with judge"""
    print(f"    Evaluating...", end=" ")
    
    prompt = create_judge_prompt(scenario, response)
    
    try:
        messages = [{"role": "user", "content": prompt}]
        judge_response, _, _ = call_openrouter(messages, judge_model, temperature=0.3, max_tokens=2000)
        parsed = parse_judge_response(judge_response)
        criteria_scores = {
            crit: float(parsed.get("criteria", {}).get(crit, {}).get("score", 0))
            for crit in RUBRIC_CRITERIA
        }
        
        overall = float(parsed.get("overall_score", 0))
        print(f"✓ Score: {overall:.1f}/100")
        
        return RubricScore(
            scenario_id=scenario.id,
            prompt_name=response.prompt_name,
            criteria_scores=criteria_scores,
            overall_score=overall,
            reasoning=parsed.get("overall_reasoning", ""),
            evaluated_at=datetime.now().isoformat()
        )
        
    except Exception as e:
        print(f"✗ Error: {e}")
        return RubricScore(
            scenario_id=scenario.id,
            prompt_name=response.prompt_name,
            criteria_scores={crit: 0.0 for crit in RUBRIC_CRITERIA},
            overall_score=0.0,
            reasoning=f"Evaluation failed: {e}",
            evaluated_at=datetime.now().isoformat()
        )
print("✓ All functions loaded")

In [ ]:
def run_scenario(scenario_num: str):
    """
    Run all prompts for a single scenario number.
    Args:
        scenario_num: Scenario number as string (e.g., '001', '002')
    Returns:
        Dict with results summary
    """
    print(f"\n{'='*60}")
    print(f"RUNNING SCENARIO #{scenario_num}")
    print(f"{'='*60}")

    print(f"\nLoading: {TEST_DATA_FILE}")
    test_data = load_test_data(TEST_DATA_FILE)
    print(f"✓ {len(test_data.prompts)} prompts, {len(test_data.scenarios)} scenarios loaded")

    scenario_id = f"scenario_{scenario_num}"
    target_scenario = None
    for s in test_data.scenarios:
        if s.id == scenario_id:
            target_scenario = s
            break
    if not target_scenario:
        print(f"\n✗ Scenario '{scenario_id}' not found!")
        print(f"Available scenarios: {[s.id for s in test_data.scenarios]}")
        return None
    print(f"\n✓ Found scenario: {target_scenario.id} - {target_scenario.character_name}")
    
    results_summary = {
        'scenario_num': scenario_num,
        'scenario_id': target_scenario.id,
        'character_name': target_scenario.character_name,
        'scores': {}
    }

    for prompt_name, prompt_template in test_data.prompts.items():
        print(f"\n{'='*60}")
        print(f"Testing Prompt: {prompt_name}")
        print(f"{'='*60}")

        response = run_multi_turn_scenario(
            target_scenario,
            prompt_template,
            prompt_name,
            BASELINE_MODEL,
            temperature=0.7
        )
        score = evaluate_scenario(target_scenario, response, JUDGE_MODEL)
        result = PromptResults(
            prompt_name=prompt_name,
            scenario_responses=[response],
            rubric_scores=[score],
            mean_score=score.overall_score,
            median_score=score.overall_score,
            std_score=0.0,
            min_score=score.overall_score,
            max_score=score.overall_score,
            criteria_means=score.criteria_scores,
            total_tokens=response.total_tokens,
            total_time=response.generation_time
        )
        output_filename = f"{prompt_name}_results_{scenario_num}.json"
        with open(OUTPUT_DIR / output_filename, 'w') as f:
            json.dump(asdict(result), f, indent=2, default=str)
        print(f"\n✓ Saved: {output_filename}")
        print(f"  Score: {score.overall_score:.2f}/100")
        results_summary['scores'][prompt_name] = score.overall_score
    print(f"\n{'='*60}")
    print(f"SCENARIO #{scenario_num} COMPLETE!")
    print(f"{'='*60}")
    return results_summary

print("✓ Main function defined - ready to run scenarios!")

In [ ]:
run_scenario("001")

In [ ]:
run_scenario("002")

In [ ]:
run_scenario("003")

In [ ]:
run_scenario("004")

In [ ]:
run_scenario("005")

In [ ]:
run_scenario("006")

In [ ]:
run_scenario("007")

In [ ]:
run_scenario("008")

In [ ]:
run_scenario("009")

In [ ]:
run_scenario("010")

In [ ]:
run_scenario("011")

In [ ]:
run_scenario("012")

In [ ]:
run_scenario("013")

In [ ]:
run_scenario("014")

In [ ]:
run_scenario("015")

In [ ]:
run_scenario("016")

In [ ]:
run_scenario("017")

In [ ]:
run_scenario("018")

In [ ]:
run_scenario("019")

In [ ]:
run_scenario("020")

In [ ]:
run_scenario("021")

In [ ]:
run_scenario("022")

In [ ]:
run_scenario("023")

In [ ]:
run_scenario("024")

In [ ]:
run_scenario("025")

In [ ]:
run_scenario("026")

In [ ]:
run_scenario("027")

In [ ]:
run_scenario("028")

In [ ]:
run_scenario("029")

In [ ]:
run_scenario("030")

In [ ]:
run_scenario("031")

In [ ]:
run_scenario("032")

In [ ]:
run_scenario("033")

In [ ]:
run_scenario("034")

In [ ]:
run_scenario("035")

In [ ]:
run_scenario("036")

In [ ]:
run_scenario("037")

In [ ]:
run_scenario("038")

In [ ]:
run_scenario("039")

In [ ]:
run_scenario("040")

In [ ]:
run_scenario("041")

In [ ]:
run_scenario("042")

In [ ]:
run_scenario("043")

In [ ]:
run_scenario("044")

In [ ]:
run_scenario("045")

In [197]:
INPUT_DIR = Path("./eqbench3_results")
OUTPUT_FILE = INPUT_DIR / "merged_comparison.json"
RUBRIC_CRITERIA = [
    "demonstrated_empathy",
    "emotional_insight",
    "contextual_appropriateness",
    "theory_of_mind",
    "response_depth"
]

In [198]:
@dataclass
class RubricScore:
    scenario_id: str
    prompt_name: str
    criteria_scores: Dict[str, float]
    overall_score: float
    reasoning: str

@dataclass
class PromptResults:
    prompt_name: str
    rubric_scores: List[RubricScore]
    mean_score: float
    median_score: float
    std_score: float
    min_score: float
    max_score: float
    criteria_means: Dict[str, float]

@dataclass
class ComparisonResult:
    prompt_results: Dict[str, PromptResults]
    baseline_model: str = "Nous Hermes 3 405B"
    judge_model: str = "Claude Sonnet 3.7"
    total_scenarios: int = 0

In [199]:
def load_and_merge_data(directory: Path) -> ComparisonResult:
    """Scans directory for numbered result files and merges them by prompt."""
    merged_data = defaultdict(list)
    baseline_model = "Unknown"
    judge_model = "Unknown"
    unique_scenarios = set()

    print(f"Scanning {directory} for numbered result files...")
    for filepath in directory.glob("*_results_*.json"):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if "prompt_name" not in data or "rubric_scores" not in data:
                continue
            p_name = data["prompt_name"]
            print(f"  Loading: {filepath.name} ({p_name})")
            for score_data in data.get("rubric_scores", []):
                if isinstance(score_data, dict):
                    score = RubricScore(
                        scenario_id=score_data.get("scenario_id"),
                        prompt_name=score_data.get("prompt_name"),
                        criteria_scores=score_data.get("criteria_scores"),
                        overall_score=float(score_data.get("overall_score")),
                        reasoning=score_data.get("reasoning")
                    )
                    merged_data[p_name].append(score)
                    unique_scenarios.add(score.scenario_id)

        except Exception as e:
            print(f"  ⚠️  Skipping {filepath.name}: {e}")
    final_results = {}
    for prompt, scores in merged_data.items():
        if not scores:
            continue
        overall_vals = [s.overall_score for s in scores]
        crit_means = {}
        for crit in RUBRIC_CRITERIA:
            vals = [s.criteria_scores.get(crit, 0) for s in scores]
            crit_means[crit] = np.mean(vals) if vals else 0.0

        results = PromptResults(
            prompt_name=prompt,
            rubric_scores=scores,
            mean_score=np.mean(overall_vals),
            median_score=np.median(overall_vals),
            std_score=np.std(overall_vals),
            min_score=np.min(overall_vals),
            max_score=np.max(overall_vals),
            criteria_means=crit_means
        )
        final_results[prompt] = results
        print(f"  ✓ {prompt}: {len(scores)} scenarios, mean={results.mean_score:.2f}")
    print(f"\n✓ Merged {len(unique_scenarios)} unique scenarios across {len(final_results)} prompts")
    return ComparisonResult(
        prompt_results=final_results,
        total_scenarios=len(unique_scenarios)
    )

In [ ]:
def generate_score_distributions(comparison: ComparisonResult):
    """1. Score distributions histogram"""
    print("\nGenerating score distributions...")
    
    prompts = list(comparison.prompt_results.keys())
    if not prompts:
        return

    num_plots = len(prompts)
    cols = 3
    rows = (num_plots + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = np.array(axes).flatten()
    
    for idx, (prompt_name, result) in enumerate(comparison.prompt_results.items()):
        scores = [s.overall_score for s in result.rubric_scores]
        axes[idx].hist(scores, bins=15, edgecolor='black', alpha=0.7)
        axes[idx].axvline(result.mean_score, color='red', linestyle='--', linewidth=2)
        axes[idx].set_title(f'{prompt_name}\nμ={result.mean_score:.1f}')
        axes[idx].set_xlabel('Score')
        axes[idx].set_ylabel('Count')
        axes[idx].set_xlim(0, 100)

    for idx in range(len(prompts), len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig(INPUT_DIR / "score_distributions.png", dpi=150)
    print("  ✓ score_distributions.png")
    plt.close()

def generate_criteria_breakdown(comparison: ComparisonResult):
    """2. Criteria breakdown grouped bar chart"""
    print("Generating criteria breakdown...")
    prompts = list(comparison.prompt_results.keys())
    x = np.arange(len(prompts))
    width = 0.15
    
    fig, ax = plt.subplots(figsize=(14, 8))
    for i, crit in enumerate(RUBRIC_CRITERIA):
        scores = [comparison.prompt_results[p].criteria_means[crit] for p in prompts]
        offset = width * (i - len(RUBRIC_CRITERIA)/2 + 0.5)
        ax.bar(x + offset, scores, width, label=crit.replace('_', ' ').title())
    
    ax.set_xlabel('Prompt')
    ax.set_ylabel('Score (0-20)')
    ax.set_title('Criteria Performance Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(prompts, rotation=45, ha='right')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(INPUT_DIR / "criteria_breakdown.png", dpi=150, bbox_inches='tight')
    print("  ✓ criteria_breakdown.png")
    plt.close()

def generate_score_variance(comparison: ComparisonResult):
    """3. Score variance with error bars"""
    print("Generating score variance...")
    
    prompts = list(comparison.prompt_results.keys())
    x = np.arange(len(prompts))
    means = [comparison.prompt_results[p].mean_score for p in prompts]
    stds = [comparison.prompt_results[p].std_score for p in prompts]
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(x, means, yerr=stds, capsize=10, alpha=0.7, edgecolor='black')
    cmap = plt.cm.RdYlGn
    norm = plt.Normalize(vmin=min(means), vmax=max(means))
    for bar, mean in zip(bars, means):
        bar.set_color(cmap(norm(mean)))
    
    ax.set_xlabel('Prompt')
    ax.set_ylabel('Mean Score')
    ax.set_title('Mean Scores with Standard Deviation')
    ax.set_xticks(x)
    ax.set_xticklabels(prompts, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(INPUT_DIR / "score_variance.png", dpi=150)
    print("  ✓ score_variance.png")
    plt.close()

def generate_radar_chart(comparison: ComparisonResult):
    """4. Radar chart for criteria comparison"""
    print("Generating radar chart...")
    prompts = list(comparison.prompt_results.keys())
    N = len(RUBRIC_CRITERIA)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    colors = plt.cm.Set2(np.linspace(0, 1, len(prompts)))
    for idx, prompt_name in enumerate(prompts):
        result = comparison.prompt_results[prompt_name]
        values = [result.criteria_means[crit] for crit in RUBRIC_CRITERIA]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=prompt_name, color=colors[idx])
        ax.fill(angles, values, alpha=0.15, color=colors[idx])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([c.replace('_', '\n').title() for c in RUBRIC_CRITERIA], size=9)
    ax.set_ylim(0, 20)
    ax.set_title('Criteria Performance Radar', size=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(INPUT_DIR / "radar_chart.png", dpi=150, bbox_inches='tight')
    print("  ✓ radar_chart.png")
    plt.close()

def analyze_introspection_quality(directory: Path) -> Dict[str, Any]:
    """5. Introspection quality analysis from numbered files"""
    print("\nAnalyzing introspection quality...")
    
    introspection_stats = defaultdict(lambda: {
        'total_turns': 0,
        'missing_self': 0,
        'missing_other': 0,
        'has_thinking': 0,
        'self_lengths': [],
        'other_lengths': []
    })
    for filepath in directory.glob("*_results_*.json"):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if "prompt_name" not in data or "scenario_responses" not in data:
                continue
            
            prompt_name = data["prompt_name"]
            
            # Extract introspection from each turn
            for response in data.get("scenario_responses", []):
                for turn in response.get("turn_responses", []):
                    stats = introspection_stats[prompt_name]
                    stats['total_turns'] += 1
                    
                    # Safely get introspection (might be missing if parsing failed)
                    introspection = turn.get("introspection")
                    if not introspection:
                        stats['missing_self'] += 1
                        stats['missing_other'] += 1
                        continue
                    
                    self_thinking = introspection.get("self_thinking", "")
                    other_thinking = introspection.get("other_thinking", "")
                    
                    if not self_thinking:
                        stats['missing_self'] += 1
                    else:
                        stats['self_lengths'].append(len(self_thinking))
                    
                    if not other_thinking:
                        stats['missing_other'] += 1
                    else:
                        stats['other_lengths'].append(len(other_thinking))
                    
                    if turn.get("has_thinking_tags", False):
                        stats['has_thinking'] += 1
        
        except Exception as e:
            continue
    
    print("\n" + "="*60)
    print("INTROSPECTION QUALITY ANALYSIS")
    print("="*60)
    
    for prompt_name in sorted(introspection_stats.keys()):
        stats = introspection_stats[prompt_name]
        total = stats['total_turns']
        
        if total == 0:
            continue
        
        print(f"\n{prompt_name}:")
        print(f"  Total turns: {total}")
        print(f"  Missing self-introspection: {stats['missing_self']} ({stats['missing_self']/total*100:.1f}%)")
        print(f"  Missing other-introspection: {stats['missing_other']} ({stats['missing_other']/total*100:.1f}%)")
        print(f"  Has <think> tags: {stats['has_thinking']} ({stats['has_thinking']/total*100:.1f}%)")
        
        if stats['self_lengths']:
            print(f"  Avg self-introspection length: {np.mean(stats['self_lengths']):.0f} chars")
        if stats['other_lengths']:
            print(f"  Avg other-introspection length: {np.mean(stats['other_lengths']):.0f} chars")
    
    return dict(introspection_stats)

In [201]:
def generate_report(comparison: ComparisonResult, introspection_stats: Dict[str, Any]):
    """Generate comprehensive markdown report"""
    print("\nGenerating comprehensive report...")
    
    report = []
    report.append("# EQBench3 Complete Evaluation Report\n\n")
    report.append(f"**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    report.append(f"**Total Scenarios**: {comparison.total_scenarios}\n")
    report.append(f"**Scoring Scale**: 0-20 per criterion (max 100 total)\n\n")
    report.append("---\n\n")
    report.append("## Score Anchoring Reference\n\n")
    report.append("- **0-4**: Poor, major deficiencies\n")
    report.append("- **5**: High school level\n")
    report.append("- **10**: Graduate level\n")
    report.append("- **15**: PhD level\n")
    report.append("- **19-20**: Top 0.001% (Rogers/Jung level)\n\n")
    report.append("## Overall Rankings\n\n")
    report.append("| Rank | Prompt | Mean Score | Std Dev | Range | Median | Avg per Criterion |\n")
    report.append("|------|--------|-----------|---------|-------|--------|------------------|\n")
    
    sorted_results = sorted(
        comparison.prompt_results.items(),
        key=lambda x: x[1].mean_score,
        reverse=True
    )
    baseline_score = None
    for rank, (prompt_name, result) in enumerate(sorted_results, 1):
        if prompt_name == "baseline":
            baseline_score = result.mean_score
        avg_criterion = result.mean_score / 5
        
        report.append(
            f"| {rank} | {prompt_name} | {result.mean_score:.2f}/100 | "
            f"{result.std_score:.2f} | {result.min_score:.1f}-{result.max_score:.1f} | "
            f"{result.median_score:.2f} | {avg_criterion:.2f}/20 |\n"
        )
    
    report.append("\n")
    report.append("## Criteria Breakdown (0-20 scale)\n\n")
    report.append("| Prompt | Empathy | Insight | Context | Theory of Mind | Depth |\n")
    report.append("|--------|---------|---------|---------|----------------|-------|\n")
    
    for prompt_name, result in sorted_results:
        report.append(f"| {prompt_name} |")
        for crit in RUBRIC_CRITERIA:
            score = result.criteria_means.get(crit, 0)
            report.append(f" {score:.2f} |")
        report.append("\n")
    report.append("\n")

    if baseline_score:
        report.append("## Improvement over Baseline\n\n")
        report.append("| Prompt | Absolute Difference | Relative Improvement |\n")
        report.append("|--------|-------------------|---------------------|\n")
        
        for prompt_name, result in sorted_results:
            if prompt_name != "baseline":
                diff = result.mean_score - baseline_score
                pct = (diff / baseline_score) * 100 if baseline_score > 0 else 0
                report.append(f"| {prompt_name} | {diff:+.2f} | {pct:+.1f}% |\n")
        report.append("\n")

    report.append("## Human Guidance Analysis\n\n")
    if "guided" in comparison.prompt_results:
        guided_result = comparison.prompt_results["guided"]
        report.append(f"**Guided Prompt Performance**: {guided_result.mean_score:.2f}/100\n\n")
        
        if baseline_score:
            diff = guided_result.mean_score - baseline_score
            pct_improvement = (diff / baseline_score) * 100 if baseline_score > 0 else 0
            report.append(f"**Improvement over baseline**: {diff:+.2f} points ({pct_improvement:+.1f}%)\n\n")
        if "baseline" in comparison.prompt_results:
            baseline_result = comparison.prompt_results["baseline"]
            report.append("### Guided vs Baseline by Criterion\n\n")
            report.append("| Criterion | Baseline | Guided | Difference |\n")
            report.append("|-----------|----------|--------|------------|\n")
            for crit in RUBRIC_CRITERIA:
                base_score = baseline_result.criteria_means.get(crit, 0)
                guided_score = guided_result.criteria_means.get(crit, 0)
                diff = guided_score - base_score
                report.append(f"| {crit.replace('_', ' ').title()} | {base_score:.2f} | {guided_score:.2f} | {diff:+.2f} |\n")
            report.append("\n")
    else:
        report.append("*Guided prompt was not tested in this run.*\n\n")
    
    report.append("## Statistical Summary\n\n")
    for prompt_name, result in sorted_results:
        report.append(f"### {prompt_name}\n")
        report.append(f"- **Scenarios tested**: {len(result.rubric_scores)}\n")
        report.append(f"- **Mean**: {result.mean_score:.2f}/100\n")
        report.append(f"- **Median**: {result.median_score:.2f}/100\n")
        report.append(f"- **Std Dev**: {result.std_score:.2f}\n")
        report.append(f"- **Range**: {result.min_score:.1f} - {result.max_score:.1f}\n")
        report.append(f"- **Coefficient of Variation**: {(result.std_score / result.mean_score * 100):.1f}%\n\n")
    
    report.append("## Generated Visualizations\n\n")
    report.append("1. `score_distributions.png` - Score histograms for each prompt\n")
    report.append("2. `criteria_breakdown.png` - Grouped bar chart of criteria performance\n")
    report.append("3. `score_variance.png` - Mean scores with error bars\n")
    report.append("4. `radar_chart.png` - Multi-criteria radar comparison\n\n")
    
    if introspection_stats:
        report.append("## Introspection Quality Summary\n\n")
        report.append("| Prompt | Total Turns | Missing Self | Missing Other | Avg Self Length | Avg Other Length |\n")
        report.append("|--------|-------------|--------------|---------------|-----------------|------------------|\n")
        
        for prompt_name in sorted(introspection_stats.keys()):
            stats = introspection_stats[prompt_name]
            total = stats['total_turns']
            if total == 0:
                continue
            missing_self_pct = f"{stats['missing_self']/total*100:.1f}%"
            missing_other_pct = f"{stats['missing_other']/total*100:.1f}%"
            avg_self = f"{np.mean(stats['self_lengths']):.0f}" if stats['self_lengths'] else "N/A"
            avg_other = f"{np.mean(stats['other_lengths']):.0f}" if stats['other_lengths'] else "N/A"
            report.append(
                f"| {prompt_name} | {total} | {missing_self_pct} | {missing_other_pct} | "
                f"{avg_self} | {avg_other} |\n"
            )
        
        report.append("\n")
    with open(INPUT_DIR / "report.md", 'w') as f:
        f.writelines(report)
    print("  ✓ report.md")

In [203]:
if __name__ == "__main__":
    print("="*60)
    print("EQBENCH3 COMPLETE RESULTS AGGREGATOR")
    print("="*60)
    if not INPUT_DIR.exists():
        print(f"\n✗ Error: Directory {INPUT_DIR} does not exist.")
        exit(1)
    merged_comparison = load_and_merge_data(INPUT_DIR)
    
    if not merged_comparison.prompt_results:
        print("\n✗ No valid data found to aggregate.")
        print("   Looking for files matching: *_results_*.json")
        exit(1)
    with open(OUTPUT_FILE, 'w') as f:
        output = {
            "baseline_model": merged_comparison.baseline_model,
            "judge_model": merged_comparison.judge_model,
            "total_scenarios": merged_comparison.total_scenarios,
            "prompt_results": {
                name: {
                    "prompt_name": result.prompt_name,
                    "mean_score": result.mean_score,
                    "median_score": result.median_score,
                    "std_score": result.std_score,
                    "min_score": result.min_score,
                    "max_score": result.max_score,
                    "criteria_means": result.criteria_means,
                    "num_scenarios": len(result.rubric_scores)
                }
                for name, result in merged_comparison.prompt_results.items()
            }
        }
        json.dump(output, f, indent=2)
    print(f"\n✓ Saved merged data: {OUTPUT_FILE}")
    print("\n" + "="*60)
    print("GENERATING ALL VISUALIZATIONS")
    print("="*60)
    
    generate_score_distributions(merged_comparison)
    generate_criteria_breakdown(merged_comparison)
    generate_score_variance(merged_comparison)
    generate_radar_chart(merged_comparison)
    introspection_stats = analyze_introspection_quality(INPUT_DIR)
    print("\n" + "="*60)
    print("GENERATING REPORT")
    print("="*60)
    generate_report(merged_comparison, introspection_stats)
    print("\n" + "="*60)
    print("AGGREGATION COMPLETE!")
    print("="*60)
    
    sorted_results = sorted(
        merged_comparison.prompt_results.items(),
        key=lambda x: x[1].mean_score,
        reverse=True
    )
    
    print("\nFinal Rankings:")
    for rank, (prompt_name, result) in enumerate(sorted_results, 1):
        print(f"{rank}. {prompt_name:20s} | {result.mean_score:6.2f} ± {result.std_score:5.2f}")
    print(f"\n✓ All results saved to: {INPUT_DIR}/")
    print("="*60)

EQBENCH3 COMPLETE RESULTS AGGREGATOR
Scanning eqbench3_results for numbered result files...
  Loading: baseline_results_001.json (baseline)
  Loading: baseline_results_002.json (baseline)
  Loading: baseline_results_003.json (baseline)
  Loading: baseline_results_004.json (baseline)
  Loading: baseline_results_005.json (baseline)
  Loading: baseline_results_006.json (baseline)
  Loading: baseline_results_007.json (baseline)
  Loading: baseline_results_008.json (baseline)
  Loading: baseline_results_009.json (baseline)
  Loading: baseline_results_010.json (baseline)
  Loading: baseline_results_011.json (baseline)
  Loading: baseline_results_012.json (baseline)
  Loading: baseline_results_013.json (baseline)
  Loading: baseline_results_014.json (baseline)
  Loading: baseline_results_015.json (baseline)
  Loading: baseline_results_016.json (baseline)
  Loading: baseline_results_017.json (baseline)
  Loading: baseline_results_018.json (baseline)
  Loading: baseline_results_019.json (baseli